In [1]:
import pandas as pd
# import os
# import sys
# sys.path.append(os.path.abspath('..'))
from neutrophil_shape.config.loader import load_config
from neutrophil_shape.CustomFunctions import DetailedBalance, utils

In [2]:
### load config stuff
config = load_config(microscope_type = 'confocal')
#set alignment
config._alignment = 'trajectory'
savedir = config.common.savedir
datadir = savedir / 'shape_data'
dbdir = savedir / 'detailed_balance'
npcs = config.common.npcs
ntrans = config.db_params.ntrans
origins = config.db_params.origins
pc_combos = config.common.pc_combos
####### load common directories and data
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)

## restrict the treatments and PCs specifically for bootstrapping
bstreats = ['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
bspcs = []#[[1,2],[4,5],[2,8]]
alldatabs = True

In [3]:
############# create all CGPSs #############
for whichpcs in pc_combos:

    if __name__ ==  '__main__':
        ########### get raw transitions and pairs ###########
        rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                FullFrame, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ########### interpolate all transitions so that only individual transitions are made ###########
        transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                rawtrans, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ############## get the counts of cells leaving 
        trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2287.3143701227186 minutes
Total time observed in this CGPS was 5813.237243932379 minutes
Total time observed in this CGPS was 1395.509183750561 minutes
Total time observed in this CGPS was 2950.0440673274597 minutes
Total time observed in this CGPS was 37.25634146087305 minutes
Total time observed in this CGPS was 1305.6016706618893 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2322.73848031811 minutes
Total time observed in this CGPS was 5879.759392289732 minutes
Total time observed in this CGPS was 1394.9636710140503 minutes
Total time observed in this CGPS was 2999.5853521093572 minutes
Total time observed in this CGPS was 35.94185713293227 minutes
Total time observed in this CGPS was 1305.437166600593 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajecto

In [ ]:

for whichpcs in pc_combos:
    ############# measure aer and cycling frequencies ###########
    #add specific scaling
    xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
    #set the origin to the actual center
    origin = origins[pc_combos.index(whichpcs)]

    ### open the raw transitions
    rawtrans = pd.read_csv(dbdir.joinpath(
            f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

    ############### measure aer and cycling frequency for the raw transitions
    #get the area scaling in x and y based on the size of the bins in the cgps
    results = []
    for i, cell in rawtrans.groupby('CellID'):
        #sort data and get continuous transitions in order
        cell, runs = utils.get_consecutive_transitions(cell)
        for r in runs:
            c = cell.iloc[r].reset_index(drop=True)
            results.append(DetailedBalance.get_area_enclosing_rate((
                c,
                config.db_params.nbins,
                xyscaling,
                origin,
                )))

    #make a dataframe and save it
    allaers = pd.concat(results, ignore_index = True)
    allaers.to_csv(dbdir.joinpath(
                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



In [ ]:
########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############
for whichpcs in pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue
    ## use a separate savedir to bootstrap using all data
    bssavestr = 'alldatabs' if alldatabs else 'separatedatabs'

    if __name__ ==  '__main__':
        #### open the transitions
        rawtrans = pd.read_csv(dbdir.joinpath(
            f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

        #merge all treatments if bootstrapping with all data
        if alldatabs:
            rawtrans.loc[:,'Treatment'] = 'alldata'

        #restrict to bootstrapped treatments if desired
        if len(bstreats)>0:
            rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]


        ############## BOOTSTRAP MANY TRAJECTORIES ##########
        bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                bssavestr, #where to save the bootstrapped dataframes
                )


        ############# open average bootstrapped currents ###################
        bsfield_sep = DetailedBalance.get_avg_current_error(
                bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,    
                bssavestr, #where to save the aggregated counts
                )


        ############# measure aer and cycling frequencies ###########
        #add specific scaling
        xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
        #set the origin to the actual center
        origin = origins[pc_combos.index(whichpcs)]

        DetailedBalance.get_aer_cf(
            bstrans,
            xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
            origin, #origin in [x bin,y bin]
            whichpcs, #which two PCs to use in the cgps [x,y]
            config,
            bssavestr, #where to save calculated aers and cfs
            )



In [ ]:
### load config stuff
config = load_config(microscope_type = 'confocal')

for ali in ['trajectory','shape','trajectory_shape']:

    #set alignment
    config._alignment = ali
    savedir = config.common.savedir
    datadir = savedir / 'shape_data'
    dbdir = savedir / 'detailed_balance'
    npcs = config.common.npcs
    ntrans = config.db_params.ntrans
    origins = config.db_params.origins
    ####### load common directories and data
    FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
    centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)

    ## restrict the treatments and PCs specifically for bootstrapping
    bstreats = []#['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
    bspcs = []#[[1,2]]
    alldatabs = True
    
    
    ############# create all CGPSs #############
    for a in range(1,npcs+1):
        for b in range(1,npcs+1):
            if a == b:
                continue
            elif dbdir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
                print('Already made this CGPS')
                continue
            else:
                #set the PCs
                whichpcs = [a,b]
                if __name__ ==  '__main__':


                    ########### get raw transitions and pairs ###########
                    rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                            FullFrame, #pandas dataframe with all of the cgps binned data
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            )

                    ########### interpolate all transitions so that only individual transitions are made ###########
                    transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                            rawtrans, #pandas dataframe with all of the cgps binned data
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            )

                    ############## get the counts of cells leaving 
                    trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                            transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            )



    ########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############
    for a in range(1,npcs+1):
        for b in range(1,npcs+1):
            #set the PCs
            whichpcs = [a,b]
            #pass if we want to restrict PCs
            if len(bspcs)>0 and whichpcs not in bspcs:
                continue
            ## use a separate savedir to bootstrap using all data
            if alldatabs:
                bssavestr = 'alldatabs'
            else:
                bssavestr = 'separatedatabs'

            if a == b:
                continue
            elif dbdir.joinpath(bssavestr, f'PC{b}-PC{a}_bootstrapped_{ntrans}_transitions.csv').exists():
                print('Already made this plot')
                continue
            else:
                if __name__ ==  '__main__':
                    #### open the transitions
                    rawtrans = pd.read_csv(dbdir.joinpath(
                        f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

                    #merge all treatments if bootstrapping with all data
                    if alldatabs:
                        rawtrans.loc[:,'Treatment'] = 'alldata'

                    #restrict to bootstrapped treatments if desired
                    if len(bstreats)>0:
                        rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]


                    ############## BOOTSTRAP MANY TRAJECTORIES ##########
                    bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                            rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            bssavestr, #where to save the bootstrapped dataframes
                            )


                    ############# open average bootstrapped currents ###################
                    bsfield_sep = DetailedBalance.get_avg_current_error(
                            bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,    
                            bssavestr, #where to save the aggregated counts
                            )


                    ############# measure aer and cycling frequencies ###########
                    #add specific scaling
                    xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
                    #set the origin to the actual center
                    center = origins[int(a-1)][int(b-(2+a-1))]

                    DetailedBalance.get_aer_cf(
                        bstrans,
                        xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                        center, #origin in [x bin,y bin]
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        config,
                        bssavestr, #where to save calculated aers and cfs
                        )


                    ############### measure aer and cycling frequency for the raw transitions
                    #get the area scaling in x and y based on the size of the bins in the cgps
                    results = []
                    for i, cell in rawtrans.groupby('CellID'):
                        #sort data and get continuous transitions in order
                        cell, runs = utils.get_consecutive_transitions(cell)
                        for r in runs:
                            c = cell.iloc[r].reset_index(drop=True)
                            results.append(DetailedBalance.get_area_enclosing_rate((
                                c,
                                config.db_params.nbins,
                                xyscaling,
                                center,
                                )))

                    #make a dataframe and save it
                    allaers = pd.concat(results, ignore_index = True)
                    allaers.to_csv(dbdir.joinpath(
                                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



KeyboardInterrupt: 

In [3]:
config.db_params.origins

[[[8, 8], [8, 8], [9, 8], [8, 8], [9, 7], [9, 8], [9, 8]],
 [[8, 8], [8, 8], [8, 8], [8, 8], [8, 8], [8, 8]],
 [[8, 8], [8, 8], [8, 8], [8, 8], [8, 8]],
 [[8, 8], [8, 8], [8, 8], [8, 8]],
 [[8, 8], [8, 8], [8, 8]],
 [[6, 8], [8, 8]],
 [[8, 8]]]